# P2. Data project: propensity score matching on the Lalonde data (Matching version)

The same analysis as `p1-lalonde-matchit.ipynb`, using the `lalonde` dataset that ships with the `Matching`
package. It is a different sample (the experimental NSW treated and control groups) with separate `black` and
`hisp` indicators. See also
https://github.com/vlaskinvlad/coursera-causality-crash-course/blob/master/causality_w3.ipynb

In [1]:
source("setup.R")
use_packages("tableone", "Matching")

data("lalonde", package = "Matching")
head(lalonde)

xvars <- c("age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75", "re78")
print(CreateTableOne(vars = xvars, strata = "treat", data = lalonde, test = FALSE), smd = TRUE)

,age,educ,black,hisp,married,nodegr,re74,re75,re78,u74,u75,treat
,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>
1,37,11,1,0,1,1,0,0,9930.05,1,1,1
2,22,9,0,1,0,1,0,0,3595.89,1,1,1
3,30,12,1,0,0,0,0,0,24909.50,1,1,1
4,27,11,1,0,0,1,0,0,7506.15,1,1,1
5,33,8,1,0,0,1,0,0,289.79,1,1,1
6,22,9,1,0,0,1,0,0,4056.49,1,1,1


                     Stratified by treat
                      0                 1                 SMD   
  n                       260               185                 
  age (mean (SD))       25.05 (7.06)      25.82 (7.16)     0.107
  educ (mean (SD))      10.09 (1.61)      10.35 (2.01)     0.141
  black (mean (SD))      0.83 (0.38)       0.84 (0.36)     0.044
  hisp (mean (SD))       0.11 (0.31)       0.06 (0.24)     0.175
  married (mean (SD))    0.15 (0.36)       0.19 (0.39)     0.094
  nodegr (mean (SD))     0.83 (0.37)       0.71 (0.46)     0.304
  re74 (mean (SD))    2107.03 (5687.91) 2095.57 (4886.62)  0.002
  re75 (mean (SD))    1266.91 (3102.98) 1532.06 (3219.25)  0.084
  re78 (mean (SD))    4554.80 (5483.84) 6349.15 (7867.40)  0.265


In [2]:
# propensity score model on the confounders only, not the outcome re78
psmodel <- glm(treat ~ age + educ + black + hisp + married + nodegr + re74 + re75,
               family = binomial(link = "logit"), data = lalonde)
summary(psmodel)
pscore <- psmodel$fitted.values


Call:
glm(formula = treat ~ age + educ + black + hisp + married + nodegr + 
    re74 + re75, family = binomial(link = "logit"), data = lalonde)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)   
(Intercept)  1.178e+00  1.056e+00   1.115  0.26474   
age          4.698e-03  1.433e-02   0.328  0.74297   
educ        -7.124e-02  7.173e-02  -0.993  0.32061   
black       -2.247e-01  3.655e-01  -0.615  0.53874   
hisp        -8.528e-01  5.066e-01  -1.683  0.09228 . 
married      1.636e-01  2.769e-01   0.591  0.55463   
nodegr      -9.035e-01  3.135e-01  -2.882  0.00395 **
re74        -3.161e-05  2.584e-05  -1.223  0.22122   
re75         6.161e-05  4.358e-05   1.414  0.15744   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 604.20  on 444  degrees of freedom
Residual deviance: 587.22  on 436  degrees of freedom
AIC: 605.22

Number of Fisher Scoring iterations: 4


## Greedy matching on the propensity score, without a caliper

Passing the outcome `Y` makes `Match` also estimate the treatment effect on the treated.

In [3]:
set.seed(931139)
psmatch <- Match(Y = lalonde$re78, Tr = lalonde$treat, M = 1, X = pscore)
summary(psmatch)

matched <- matched_rows(psmatch, lalonde)
print(CreateTableOne(vars = xvars, strata = "treat", data = matched, test = FALSE), smd = TRUE)


Estimate...  2624.3 
AI SE......  802.19 
T-stat.....  3.2714 
p.val......  0.0010702 

Original number of observations..............  445 
Original number of treated obs...............  185 
Matched number of observations...............  185 
Matched number of observations  (unweighted).  344 



                     Stratified by treat
                      0                 1                 SMD   
  n                       344               344                 
  age (mean (SD))       24.47 (6.47)      24.62 (6.90)     0.023
  educ (mean (SD))      10.19 (1.49)      10.37 (1.64)     0.115
  black (mean (SD))      0.92 (0.28)       0.89 (0.31)     0.088
  hisp (mean (SD))       0.03 (0.17)       0.04 (0.20)     0.063
  married (mean (SD))    0.12 (0.33)       0.15 (0.36)     0.076
  nodegr (mean (SD))     0.83 (0.38)       0.78 (0.41)     0.110
  re74 (mean (SD))    1295.89 (4302.81) 1780.12 (4193.15)  0.114
  re75 (mean (SD))    1320.29 (3463.59) 1132.30 (2549.89)  0.062
  re78 (mean (SD))    3624.86 (4432.07) 5600.71 (7086.87)  0.334


## With a caliper of 0.1, and the outcome analysis

In [4]:
set.seed(931139)
psmatch <- Match(Y = lalonde$re78, Tr = lalonde$treat, M = 1, X = pscore, caliper = 0.1)
summary(psmatch)

matched <- matched_rows(psmatch, lalonde)
diffy <- matched$re78[matched$treat == 1] - matched$re78[matched$treat == 0]
mean(diffy)
t.test(diffy)


Estimate...  2561.6 
AI SE......  783.34 
T-stat.....  3.2701 
p.val......  0.0010752 

Original number of observations..............  445 
Original number of treated obs...............  185 
Matched number of observations...............  181 
Matched number of observations  (unweighted).  340 

Caliper (SDs)........................................   0.1 
Number of obs dropped by 'exact' or 'caliper'  4 



[1] 1934.842


	One Sample t-test

data:  diffy
t = 4.3535, df = 339, p-value = 1.777e-05
alternative hypothesis: true mean is not equal to 0
95 percent confidence interval:
 1060.640 2809.045
sample estimates:
mean of x 
 1934.842 
